# Лабораторная работа 1

Machine Unlearning

## Сбор публикаций

импорт библиотек

In [47]:
import re
import time
import feedparser
import pandas as pd
import requests

формирование запроса:
- устанавливаю период в котором ищем нужные публикации
- ищу в названии статьи/аннотации machine unlearning

так же заранее завела константы которые будут использоваться дальше:
- api для запроса
- размер странцы
- минимальное значение количества собранных статей после для проверки
- время для паузы после запросов (тк в документации написано что между запросами должно быть не менее 3 секунд - https://info.arxiv.org/help/api/tou.html#rate-limits:~:text=When%20using%20the%20legacy%20APIs%20(including%20OAI%2DPMH%2C%20RSS%2C%20and%20the%20arXiv%20API)%2C%20make%20no%20more%20than%20one%20request%20every%20three%20seconds%2C%20and%20limit%20requests%20to%20a%20single%20connection%20at%20a%20time - взяла с запасом!)

In [48]:
API_URL = "https://export.arxiv.org/api/query"
DATE_FROM = "2020-01-01T00:00:00Z"
DATE_TO = "2026-09-20T21:47:00Z"
MIN_PUBLICATIONS = 300
PAGE_SIZE = 100
REQUEST_PAUSE = 3.3

start_time = pd.Timestamp(DATE_FROM)
end_time = pd.Timestamp(DATE_TO)
SEARCH_QUERY = (
    '(ti:"machine unlearning" OR abs:"machine unlearning") '
    f'AND submittedDate:[{start_time.strftime("%Y%m%d%H%M")} '
    f'TO {end_time.strftime("%Y%m%d%H%M")}]'
)

выгрузка статей

arXiv дает ответ в формате Atom/XML (в отличие от OpenAlex - там ответ в формате json), поэтому необходимо использовать feedparser.parse()

ниже описана функция, которая получает только 1 страницу - в ячейке после нее в цикле получаю нужно количество статей


In [49]:
def fetch_page(start):
    params = {
        "search_query": SEARCH_QUERY,
        "start": start,
        "max_results": PAGE_SIZE,
        "sortBy": "submittedDate",
        "sortOrder": "ascending",
    }
    time.sleep(REQUEST_PAUSE)
    response = requests.get(API_URL, params=params, timeout=60)
    response.raise_for_status()
    feed = feedparser.parse(response.content)
    total = int(feed.feed["opensearch_totalresults"])
    return feed.entries, total


In [50]:
records = []
start = 0
total = None
raw_df = df = None

while total is None or start < total:
    entries, page_total = fetch_page(start)
    if total is None:
        total = page_total
        if total < MIN_PUBLICATIONS:
            raise ValueError("найдено меньше 300 работ")

    for entry in entries:
        version_id = entry.get("id", "").rstrip("/").rsplit("/", 1)[-1]
        arxiv_id = re.sub(r"v\d+$", "", version_id)
        records.append({
            "arxiv_id": arxiv_id,
            "version_id": version_id,
            "title": entry.get("title", ""),
            "authors": [author.get("name", "") for author in entry.get("authors", [])],
            "abstract": entry.get("summary", ""),
            "published": entry.get("published", ""),
            "updated": entry.get("updated", ""),
            "url": f"https://arxiv.org/abs/{arxiv_id}",
            "categories": [tag.get("term", "") for tag in entry.get("tags", [])],
        })

    start += len(entries)
    print(f"получено {len(records)} / {total}")


получено 100 / 854
получено 200 / 854
получено 300 / 854
получено 400 / 854
получено 500 / 854
получено 600 / 854
получено 700 / 854
получено 800 / 854
получено 854 / 854


подготовка и проверка таблицы

нужно:
- исправить пробелы/переносы
- перевести даты в UTC
- удалить неполные записи
- удалить статьи вне периода
- не должно быть дублей одной и той же статьи

In [51]:
raw_df = pd.DataFrame(records)
df = raw_df.copy()

def compact_spaces(text):
    return " ".join(str(text or "").split())

for column in ("title", "abstract"):
    df[column] = df[column].map(compact_spaces)
df["authors"] = df["authors"].map(
    lambda names: [compact_spaces(name) for name in names if compact_spaces(name)]
)
for column in ("published", "updated"):
    df[column] = pd.to_datetime(df[column], utc=True, errors="coerce")

valid = (
    df["arxiv_id"].str.fullmatch(r"\d{4}\.\d{4,5}")
    & df["title"].ne("")
    & df["abstract"].ne("")
    & df["authors"].map(len).gt(0)
    & df["published"].between(start_time, end_time)
)
df = (
    df.loc[valid]
    .sort_values(["updated", "version_id"], na_position="first")
    .drop_duplicates("arxiv_id", keep="last")
    .sort_values(["published", "arxiv_id"])
    .reset_index(drop=True)
)
print(f"получено: {len(raw_df)}; исключено: {len(raw_df) - len(df)}; осталось: {len(df)}.")

получено: 854; исключено: 0; осталось: 854.


In [52]:
assert len(df) >= MIN_PUBLICATIONS, "меньше 300 публикаций"
assert df["arxiv_id"].is_unique, "есть повторные ID"
assert df["title"].str.strip().ne("").all(), "есть пустые названия"
assert df["abstract"].str.strip().ne("").all(), "есть пустые аннотации"
assert df["authors"].map(lambda names: bool(names) and all(names)).all(), "не заполнены авторы"
assert df["published"].between(start_time, end_time).all(), "есть даты вне периода"
print("все хорошо:)")

все хорошо:)


и пример что собралось

In [53]:
display(df[["title", "published", "url"]].sample(5, random_state=42))
print(
    f"уникальных публикаций: {len(df)}\n"
    f"даты публикаций: {df['published'].min().date()} — "f"{df['published'].max().date()}\n"
)

,title,published,url
66,GIF: A General Graph Unlearning Strategy via Influence Function,2023-04-06 03:02:54+00:00,https://arxiv.org/abs/2304.02835
434,Do LLMs Really Forget? Evaluating Unlearning with Knowledge Correlation and Confidence Awareness,2025-06-06 04:35:19+00:00,https://arxiv.org/abs/2506.05735
198,RKLD: Reverse KL-Divergence-based Knowledge Distillation for Unlearning Personal Information in Large Lang...,2024-06-04 05:51:43+00:00,https://arxiv.org/abs/2406.01983
212,Towards Efficient Target-Level Machine Unlearning Based on Essential Graph,2024-06-16 14:17:13+00:00,https://arxiv.org/abs/2406.10954
792,SCRUB-FL: Sanitizing and Cleansing Representations via Unlearning of Backdoors,2026-06-21 22:34:34+00:00,https://arxiv.org/abs/2606.22700


уникальных публикаций: 854
даты публикаций: 2020-02-07 — 2026-09-16

